# 04 - Vivienda y condiciones habitacionales (EPH)

Análisis de las condiciones de vivienda de los hogares a partir de la EPH (INDEC), usando
los parquets compilados por **00_preparacion_bases**.

**Contenido:**
1. Tipo de vivienda y régimen de tenencia (`IV1`, `II7`), último trimestre.
2. Acceso a servicios: agua, baño, desagüe (`IV6`, `IV8`, `IV11`).
3. Hacinamiento (personas por cuarto = `IX_TOT` / `II1`).
4. Evolución temporal de indicadores de déficit habitacional.

> Las variables son de la **base Hogar**. Para no contar un hogar varias veces, se toma
> **un registro por hogar** (el jefe/a, `CH03 == 1`). Se pondera con `PONDERA`.
> Ver `.claude/memoria_EPH.md` para los códigos de cada variable.

## 1. Setup (Colab)

In [ ]:
import sys, os, shutil, glob

REPO_URL = "https://github.com/santiagoriverti/analisis_EPH.git"
REPO_DIR = "/content/analisis_EPH"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROCESSED = "/content/drive/MyDrive/carga_EPH/processed"
PROCESSED_DIR = "/content/processed_local"
os.makedirs(PROCESSED_DIR, exist_ok=True)
for src in glob.glob(os.path.join(DRIVE_PROCESSED, "eph_T*.parquet")):
    dst = os.path.join(PROCESSED_DIR, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst)
n = len(glob.glob(os.path.join(PROCESSED_DIR, "eph_T*.parquet")))
print(f"Parquets locales listos en {PROCESSED_DIR}: {n} trimestres")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_panel, list_available_quarters, _period_tag

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 6)

quarters = list_available_quarters()
ULTIMO = quarters[-1]
print("Último trimestre:", _period_tag(*ULTIMO))

def cargar_hogares(columnas, quarters=None):
    """Carga columnas de hogar tomando 1 registro por hogar (jefe/a, CH03==1).

    Fuerza a numérico las variables pedidas (al concatenar pueden quedar como texto).
    """
    cols = list(dict.fromkeys(["CH03", "PONDERA", *columnas]))
    df = load_panel(columns=cols, quarters=quarters, out_dir=PROCESSED_DIR)
    df = df[df["CH03"] == 1].copy()  # un registro por hogar
    for c in [*columnas, "PONDERA"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def share_ponderado(df, col, etiquetas):
    """% ponderado por categoría de `col`, con etiquetas legibles."""
    s = df.dropna(subset=[col]).groupby(col)["PONDERA"].sum()
    s.index = s.index.map(lambda c: etiquetas.get(int(c), str(int(c))))
    return (s / s.sum() * 100)

## 2. Tipo de vivienda y régimen de tenencia (último trimestre)

In [ ]:
TIPO_VIV = {1: "Casa", 2: "Departamento", 3: "Pieza inquilinato",
            4: "Pieza hotel/pensión", 5: "Local no const.", 6: "Otros"}
TENENCIA = {1: "Propietario (viv+terreno)", 2: "Propietario (solo viv)",
            3: "Inquilino", 4: "Ocup. pago impuestos", 5: "Ocup. relación dependencia",
            6: "Ocup. gratuito", 7: "Ocup. de hecho", 8: "En sucesión", 9: "Otra"}

hog = cargar_hogares(["IV1", "II7"], quarters=[ULTIMO])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
share_ponderado(hog, "IV1", TIPO_VIV).sort_values().plot(kind="barh", color="#4C72B0", ax=ax1)
ax1.set_title(f"Tipo de vivienda - {_period_tag(*ULTIMO)}")
ax1.set_xlabel("% de hogares")

share_ponderado(hog, "II7", TENENCIA).sort_values().plot(kind="barh", color="#55A868", ax=ax2)
ax2.set_title(f"Régimen de tenencia - {_period_tag(*ULTIMO)}")
ax2.set_xlabel("% de hogares")
plt.tight_layout()
plt.show()

## 3. Acceso a servicios (último trimestre)

Agua (`IV6`), tenencia de baño (`IV8`) y tipo de desagüe (`IV11`).

In [ ]:
AGUA = {1: "Cañería dentro vivienda", 2: "Fuera viv., dentro terreno", 3: "Fuera del terreno"}
DESAGUE = {1: "Red pública (cloaca)", 2: "Cámara séptica + pozo", 3: "Solo pozo ciego",
           4: "Hoyo/excavación"}

serv = cargar_hogares(["IV6", "IV11"], quarters=[ULTIMO])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
share_ponderado(serv, "IV6", AGUA).sort_values().plot(kind="barh", color="#4C72B0", ax=ax1)
ax1.set_title(f"Acceso al agua - {_period_tag(*ULTIMO)}")
ax1.set_xlabel("% de hogares")

share_ponderado(serv, "IV11", DESAGUE).sort_values().plot(kind="barh", color="#C44E52", ax=ax2)
ax2.set_title(f"Tipo de desagüe del baño - {_period_tag(*ULTIMO)}")
ax2.set_xlabel("% de hogares")
plt.tight_layout()
plt.show()

## 4. Hacinamiento

Personas por cuarto = `IX_TOT` (miembros del hogar) / `II1` (ambientes de uso exclusivo).
Criterio INDEC: **hacinamiento crítico** si hay más de 3 personas por cuarto.

In [ ]:
hac = cargar_hogares(["IX_TOT", "II1"], quarters=[ULTIMO])
hac = hac[(hac["II1"] > 0) & hac["IX_TOT"].notna()].copy()
hac["pers_cuarto"] = hac["IX_TOT"] / hac["II1"]

cat = pd.cut(hac["pers_cuarto"], bins=[0, 1, 2, 3, np.inf],
             labels=["≤1", "1-2", "2-3", ">3 (crítico)"], right=True)
dist = hac.groupby(cat, observed=True)["PONDERA"].sum()
dist = (dist / dist.sum() * 100)

ax = dist.plot(kind="bar", color="#8172B3")
ax.set_xlabel("Personas por cuarto")
ax.set_ylabel("% de hogares")
ax.set_title(f"Hacinamiento - {_period_tag(*ULTIMO)}")
for i, v in enumerate(dist):
    ax.text(i, v + 0.4, f"{v:.1f}%", ha="center")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

critico = dist.get(">3 (crítico)", 0)
print(f"Hogares con hacinamiento crítico (>3 pers/cuarto): {critico:.1f}%")

## 5. Evolución del déficit habitacional (serie 2017-2025)

Tres indicadores por trimestre (% de hogares):
- Sin agua por cañería dentro de la vivienda (`IV6` != 1).
- Sin desagüe a red cloacal (`IV11` != 1).
- Con hacinamiento crítico (>3 personas por cuarto).

In [ ]:
todo = cargar_hogares(["IV6", "IV11", "IX_TOT", "II1"])

def deficit(g):
    w = g["PONDERA"]
    tot = w.sum()
    sin_agua = w[g["IV6"] != 1].sum() / tot * 100
    sin_cloaca = w[g["IV11"] != 1].sum() / tot * 100
    val = g[(g["II1"] > 0) & g["IX_TOT"].notna()].copy()
    val["pc"] = val["IX_TOT"] / val["II1"]
    hac_crit = val.loc[val["pc"] > 3, "PONDERA"].sum() / val["PONDERA"].sum() * 100
    return pd.Series({"sin_agua_caneria": sin_agua, "sin_cloaca": sin_cloaca,
                      "hacinamiento_critico": hac_crit})

evol = todo.groupby(["ANIO", "TRIMESTRE"]).apply(deficit, include_groups=False)
evol.index = [f"{y}T{p}" for y, p in evol.index]

ax = evol.plot(marker="o", ms=4)
ax.set_ylabel("% de hogares")
ax.set_title("Indicadores de déficit habitacional (EPH, ponderado)")
ax.set_xticks(range(len(evol.index)))
ax.set_xticklabels(evol.index, rotation=90)
ax.legend()
plt.tight_layout()
plt.show()
evol.round(1)

---
Notebook sobre los parquets de `00_preparacion_bases`. Variables de la base Hogar,
tomando 1 registro por hogar (jefe/a) y ponderando con `PONDERA`.